# Gated Consensus Ensemble — Option B: Tightened Thresholds

Push per-method thresholds tight enough that methods which don't produce
good visual fits get naturally silenced (0 qualifying genes) for
problematic clusters. The tradeoff: some methods may also get silenced
on clusters where they could still contribute.

**Only LT data.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os, pickle

%matplotlib inline

results = pickle.load(open('metrics_results.pkl', 'rb'))
constraint_patternsLT = results['constraint_patternsLT']
all_methodsLT = results['all_methodsLT']
method_names = list(all_methodsLT.keys())

print('Methods:', method_names)
print('Clusters:', list(constraint_patternsLT.keys()))

## 1. Tightened Thresholds

Each method gets a **(direction, value)** pair:
- `('>', 0.999)` means the gene's score must be **above** 0.999 to qualify
- `('<', 0.05)` means the gene's score must be **below** 0.05 to qualify

These are intentionally much tighter than the base gated notebook.
The goal is to let bad-fit methods self-silence rather than contributing
noise to the ensemble.

In [ ]:
# ============================================================
#  OPTION B — TIGHTENED THRESHOLDS
#  From visual sweep analysis. Much stricter than base notebook.
#  Edit these and re-run cells below to see the effect.
# ============================================================
THRESHOLDS = {
    'Pearson':  ('>', 0.999),   # very tight — silences on clusterThree/Four
    'DTW':      ('<', 0.05),    # tight — silences on clusterTwo and Four
    'Cosine':   ('>', 0.9999),  # effectively silenced everywhere
    'Frechet':  ('<', 0.01),    # tight — only a few genes pass anywhere
    'MSE':      ('<', 0.0005),  # very tight — clusterOne: ~11 genes, clusterTwo: ~1
    'sMAPE':    ('<', 0.05),    # tight — clusterTwo: ~13, clusterThree: ~11
}

TOP_N = 20

print('Option B — tightened thresholds:')
for m, (op, val) in THRESHOLDS.items():
    print(f'  {m:>8}: score {op} {val}')

## 2. Qualifying Gene Counts

In [ ]:
def get_qualifying_genes(methods_dict, cluster_name, pattern_name, thresholds):
    """For each method, return the genes that pass its threshold.

    Returns dict: method_name -> Series of scores (only qualifying genes),
    sorted best-first.
    """
    qualifying = {}
    for mname, (scores, ascending) in methods_dict.items():
        if mname not in thresholds:
            continue
        op, val = thresholds[mname]
        s = scores[cluster_name][pattern_name]
        if op == '>':
            mask = s > val
        elif op == '<':
            mask = s < val
        elif op == '>=':
            mask = s >= val
        elif op == '<=':
            mask = s <= val
        else:
            raise ValueError(f'Unknown operator: {op}')
        qualifying[mname] = s[mask].sort_values(ascending=ascending)
    return qualifying


# Show qualifying gene counts per method x cluster
print(f'{"=" * 70}')
print(f'LT DATA — qualifying gene counts (Option B thresholds)')
print(f'{"=" * 70}')
for cluster_name in constraint_patternsLT:
    q = get_qualifying_genes(all_methodsLT, cluster_name, 'constraint', THRESHOLDS)
    print(f'\n  {cluster_name}:')
    for mname in method_names:
        n = len(q.get(mname, []))
        op, val = THRESHOLDS[mname]
        tag = ' ** SILENCED **' if n == 0 else ''
        print(f'    {mname:>8} ({op}{val}): {n:>6,} genes{tag}')

## 3. Qualifying Genes Per Method Per Cluster

Side-by-side panels for each cluster showing only the active (non-silenced)
methods and their qualifying genes overlaid on the constraint pattern.

In [ ]:
def plot_qualifying_genes(methods_dict, pat_dict, thresholds, max_plot=50):
    """For each cluster, plot side-by-side panels of qualifying genes
    per active method overlaid on the constraint pattern (LT only)."""
    x = [0, 3, 6, 9]

    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f'analysis data/gene_countsLT/{cluster_name}_annotated.csv'
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x

        for pattern_name, pattern_vals in pattern_dict.items():
            q = get_qualifying_genes(methods_dict, cluster_name,
                                     pattern_name, thresholds)
            active = [m for m in method_names if len(q.get(m, [])) > 0]
            silenced = [m for m in method_names if len(q.get(m, [])) == 0]
            n_active = len(active)

            if n_active == 0:
                print(f'{cluster_name}: ALL METHODS SILENCED')
                continue

            fig, axes = plt.subplots(1, n_active,
                                     figsize=(5 * n_active, 4.5),
                                     squeeze=False)

            for i, mname in enumerate(active):
                ax = axes[0, i]
                genes = q[mname].index.tolist()
                n_genes = len(genes)
                plotted = 0
                for gene in genes[:max_plot]:
                    if gene in gene_df.index:
                        ax.plot(x, gene_df.loc[gene], color='steelblue',
                                alpha=0.3, linewidth=0.8)
                        plotted += 1
                ax.plot(x, pattern_vals, color='crimson', linewidth=2.5,
                        linestyle='--')
                op, val = thresholds[mname]
                ax.set_title(f'{mname} ({op}{val})\n{n_genes:,} genes qualify',
                             fontsize=10)
                ax.set_xlabel('Timepoint')
                ax.set_xticks(x)
                if i == 0:
                    ax.set_ylabel('Gene counts')
                ax.grid(alpha=0.2)

            sil_str = ', '.join(silenced) if silenced else 'none'
            fig.suptitle(f'{cluster_name} (LT) — silenced: {sil_str}',
                         fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.show()


plot_qualifying_genes(all_methodsLT, constraint_patternsLT, THRESHOLDS)

## 4. Gated Consensus Ensemble

A gene gets a vote from a method **only if it passes that method's
threshold**. Methods with 0 qualifying genes are automatically silenced.
A gene needs **at least 1 vote** to appear in the final list.

In [ ]:
def gated_consensus(methods_dict, cluster_name, pattern_name,
                    thresholds, top_n=20):
    """Score-gated ensemble. A gene gets a vote from a method only if
    it passes that method's threshold.

    Returns (result_df, active_methods, silenced_methods).
    result_df has columns: votes, max_possible, mean_rank, qualified_in.
    """
    q = get_qualifying_genes(methods_dict, cluster_name,
                             pattern_name, thresholds)
    active = [m for m in method_names if len(q.get(m, [])) > 0]
    silenced = [m for m in method_names if len(q.get(m, [])) == 0]

    if len(active) == 0:
        return pd.DataFrame(), active, silenced

    # Build rank df for active methods only
    rank_df = pd.DataFrame()
    for mname in active:
        scores, ascending = methods_dict[mname]
        s = scores[cluster_name][pattern_name]
        rank_df[mname] = s.rank(ascending=ascending, method='min')

    # A gene gets a vote from a method only if it's in that method's
    # qualifying set
    vote_df = pd.DataFrame(index=rank_df.index)
    for mname in active:
        qualifying_genes = set(q[mname].index)
        vote_df[mname] = rank_df.index.isin(qualifying_genes).astype(int)

    votes = vote_df.sum(axis=1)
    mean_rank = rank_df[active].mean(axis=1)

    # Which methods each gene qualified in
    qualified_in = vote_df.apply(
        lambda row: ', '.join([m for m in active if row[m] == 1]),
        axis=1
    )

    result = pd.DataFrame({
        'votes': votes,
        'max_possible': len(active),
        'mean_rank': mean_rank,
        'qualified_in': qualified_in,
    })
    # Only keep genes with at least 1 vote
    result = result[result['votes'] > 0]
    result = result.sort_values(['votes', 'mean_rank'],
                                ascending=[False, True])

    return result.head(top_n), active, silenced

## 5. Results Table

Per-cluster: active/silenced methods, top 20 genes with vote counts
and which methods they qualified in.

In [ ]:
print('GATED CONSENSUS — OPTION B (Tightened Thresholds, LT Data)')
print('=' * 85)

for cluster_name in constraint_patternsLT:
    result, active, silenced = gated_consensus(
        all_methodsLT, cluster_name, 'constraint', THRESHOLDS, top_n=TOP_N
    )
    print(f'\n--- {cluster_name} ---')
    print(f'  Active:   {len(active)}/{len(method_names)} — {", ".join(active) if active else "NONE"}')
    if silenced:
        print(f'  Silenced: {", ".join(silenced)}')

    if result.empty:
        print('  NO QUALIFYING GENES — all methods silenced at these thresholds')
        continue

    print(f'\n  {"Rank":<6}{"Gene":<22}{"Votes":>7}  {"Mean Rank":>10}  Qualified in')
    print(f'  {"-" * 78}')
    for i, (gene, row) in enumerate(result.iterrows(), 1):
        v = int(row['votes'])
        mx = int(row['max_possible'])
        mr = row['mean_rank']
        qi = row['qualified_in']
        print(f'  {i:<6}{gene:<22}{v:>3}/{mx}  {mr:>10.1f}  {qi}')

## 6. Gated Ensemble Plots

Gene opacity scales with vote count: darker lines = more methods agree
that this gene is a strong match.

In [ ]:
def plot_gated_ensemble(methods_dict, pat_dict, thresholds, top_n=20):
    """Plot gated consensus top-N genes with opacity by vote count.
    LT data only."""
    x = [0, 3, 6, 9]

    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f'analysis data/gene_countsLT/{cluster_name}_annotated.csv'
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x

        for pattern_name, pattern_vals in pattern_dict.items():
            result, active, silenced = gated_consensus(
                methods_dict, cluster_name, pattern_name,
                thresholds, top_n=top_n
            )
            if result.empty:
                print(f'{cluster_name}: no qualifying genes — skipping plot')
                continue

            max_votes = len(active)
            fig, ax = plt.subplots(figsize=(10, 6))

            for gene in result.index:
                if gene in gene_df.index:
                    v = result.loc[gene, 'votes']
                    alpha = 0.2 + 0.6 * (v / max_votes)
                    ax.plot(x, gene_df.loc[gene], color='steelblue',
                            alpha=alpha, linewidth=1)

            ax.plot(x, pattern_vals, color='crimson', linewidth=2.5,
                    linestyle='--')

            sil_str = ', '.join(silenced) if silenced else 'none'
            act_str = ', '.join(active)
            n_shown = min(top_n, len(result))
            title = (f'Gated Consensus (Option B) | {cluster_name} (LT) | '
                     f'Top {n_shown} genes\n'
                     f'Active: {act_str} | Silenced: {sil_str}')
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('Timepoint')
            ax.set_ylabel('Gene counts')
            ax.set_xticks(x)

            legend_elements = [
                Line2D([0], [0], color='crimson', linewidth=2.5,
                       linestyle='--', label='Constraint pattern'),
                Line2D([0], [0], color='steelblue', alpha=0.8,
                       label=f'High consensus (>={max_votes-1}/{max_votes} votes)'),
                Line2D([0], [0], color='steelblue', alpha=0.3,
                       label='Low consensus (1 vote)'),
            ]
            ax.legend(handles=legend_elements, loc='upper right')
            plt.tight_layout()
            plt.show()


plot_gated_ensemble(all_methodsLT, constraint_patternsLT, THRESHOLDS,
                    top_n=TOP_N)